# Exploratory Data Analysis (EDA)

This notebook provides a comprehensive EDA for the DIGIT Federated Recommenders dataset.

## Usage

1. **Set dataset path**: Uses `DATASET` and `IID_TYPE` from constants (or update `DATASET_PATH`)
2. **Run all cells**: Execute cells sequentially
3. **Customize**: Modify visualizations or add analysis as needed

## What This Notebook Covers

- **Dataset Overview**: Basic statistics, shape, missing values
- **Feature Distributions**: Demographics, symptoms, anatomical features
- **Risk Distributions**: Target variable distributions and correlations
- **Client Heterogeneity**: IID/non-IID metrics (JSD, CV, entropy, imbalance ratios)
- **Visualizations**: Key plots for understanding data distribution

  
Disclaimer: **This notebook is AI generated and not actively maintained. Feel free to adjust to your needings**

In [ ]:
# Configuration
import sys
from pathlib import Path

# Add project root to path
project_root = Path().resolve().parent
sys.path.insert(0, str(project_root))

# Dataset path - Uses constants from project
from digit_fr.ml.constants import DATASET, IID_TYPE
from digit_fr.core.paths import root_path

DATASET_PATH = root_path('data', 'raw', f'fed_recommenders_synthetic_dataset_{DATASET}_{IID_TYPE}.csv')

print(f"Dataset: {DATASET}")
print(f"IID Type: {IID_TYPE}")
print(f"Dataset path: {DATASET_PATH}")

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial.distance import jensenshannon

# Project imports
from digit_fr.ml.constants import RISK_NAMES
from digit_fr.data_generation.partitioning.niid_bench_partitioning import (
    compute_partition_heterogeneity_metrics,
    compute_quantity_skew_metrics,
    get_label_distribution_stats,
    get_client_sizes
)

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("Imports successful!")

In [ ]:
# Load dataset
df = pd.read_csv(DATASET_PATH)

print(f"Dataset shape: {df.shape}")
print(f"\nColumns ({len(df.columns)}):")
print(df.columns.tolist())
print(f"\nFirst few rows:")
df.head()

In [ ]:
# Basic Statistics
print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)

print(f"\nTotal samples: {len(df):,}")
print(f"Total features: {len(df.columns)}")
print(f"Number of clients: {df['Client'].nunique() if 'Client' in df.columns else 'N/A'}")

print(f"\nMissing values:")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing Count', ascending=False)
if len(missing_df) > 0:
    print(missing_df)
else:
    print("No missing values!")

print(f"\nData types:")
print(df.dtypes.value_counts())

In [ ]:
# Risk Distributions
print("=" * 60)
print("RISK DISTRIBUTIONS")
print("=" * 60)

risk_cols = [f"Risk_{name}" for name in RISK_NAMES]
risk_probs = [f"Risk_{name}_Prob" for name in RISK_NAMES]

print("\nBinary Risk Outcomes:")
for risk_col in risk_cols:
    if risk_col in df.columns:
        counts = df[risk_col].value_counts().sort_index()
        pct = (counts / len(df) * 100).round(2)
        print(f"\n{risk_col}:")
        for val, count in counts.items():
            print(f"  {val}: {count:,} ({pct[val]:.2f}%)")

print("\nRisk Probabilities (Summary Statistics):")
for prob_col in risk_probs:
    if prob_col in df.columns:
        print(f"\n{prob_col}:")
        print(f"  Mean: {df[prob_col].mean():.4f}")
        print(f"  Std:  {df[prob_col].std():.4f}")
        print(f"  Min:  {df[prob_col].min():.4f}")
        print(f"  Max:  {df[prob_col].max():.4f}")

In [ ]:
# Visualize Risk Distributions
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, risk_name in enumerate(RISK_NAMES):
    risk_col = f"Risk_{risk_name}"
    prob_col = f"Risk_{risk_name}_Prob"
    
    ax = axes[idx]
    
    if risk_col in df.columns:
        # Binary distribution
        counts = df[risk_col].value_counts().sort_index()
        ax.bar(counts.index, counts.values, alpha=0.7, color=['skyblue', 'coral'])
        ax.set_xlabel('Outcome')
        ax.set_ylabel('Count')
        ax.set_title(f'{risk_name}\nBinary Distribution')
        ax.set_xticks([0, 1])
        ax.set_xticklabels(['No', 'Yes'])
        
        # Add probability distribution if available
        if prob_col in df.columns:
            ax2 = ax.twinx()
            ax2.hist(df[prob_col], bins=50, alpha=0.3, color='green', label='Probability')
            ax2.set_ylabel('Probability Distribution', color='green')
            ax2.tick_params(axis='y', labelcolor='green')

plt.tight_layout()
plt.show()

In [ ]:
# Client Heterogeneity Metrics (IID/non-IID Analysis)
print("=" * 60)
print("CLIENT HETEROGENEITY METRICS")
print("=" * 60)

if 'Client' in df.columns:
    # Determine label column for heterogeneity analysis
    label_column = None
    if 'Risk_Category_Composite' in df.columns:
        label_column = 'Risk_Category_Composite'
    elif any(col.startswith('Risk_Category_') for col in df.columns):
        label_column = [col for col in df.columns if col.startswith('Risk_Category_')][0]
    elif any(col.startswith('Risk_') and not col.endswith('_Prob') for col in df.columns):
        label_column = [col for col in df.columns if col.startswith('Risk_') and not col.endswith('_Prob')][0]
    
    if label_column:
        print(f"\nUsing label column: {label_column}")
        
        # Label Distribution Heterogeneity
        print("\n--- Label Distribution Heterogeneity ---")
        heterogeneity_metrics = compute_partition_heterogeneity_metrics(df, label_column, "Client")
        for metric, value in heterogeneity_metrics.items():
            print(f"{metric}: {value:.4f}")
        
        # Quantity Skew Metrics
        print("\n--- Quantity Skew Metrics ---")
        quantity_metrics = compute_quantity_skew_metrics(df, "Client")
        for metric, value in quantity_metrics.items():
            if isinstance(value, float):
                print(f"{metric}: {value:.4f}")
            else:
                print(f"{metric}: {value}")
        
        # Compute JSD metrics
        print("\n--- Jensen-Shannon Divergence (JSD) ---")
        
        # Get label distributions per client
        stats = get_label_distribution_stats(df, label_column, "Client")
        all_labels = sorted(set(label for client_stats in stats.values() for label in client_stats.keys()))
        
        # Global distribution
        global_counts = df[label_column].value_counts().sort_index()
        global_total = len(df)
        global_probs = np.array([global_counts.get(label, 0) / global_total for label in all_labels])
        
        # Client distributions
        client_probs = []
        client_ids = sorted(stats.keys())
        
        for client_id in client_ids:
            client_stats = stats[client_id]
            total = sum(client_stats.values())
            if total > 0:
                probs = np.array([client_stats.get(label, 0) / total for label in all_labels])
                client_probs.append(probs)
        
        # Compute JSD: client vs global
        jsd_client_global = []
        for probs in client_probs:
            jsd = jensenshannon(probs, global_probs)
            jsd_client_global.append(jsd)
        
        print(f"Client-Global JSD (mean): {np.mean(jsd_client_global):.4f}")
        print(f"Client-Global JSD (max):  {np.max(jsd_client_global):.4f}")
        print(f"Client-Global JSD (min):  {np.min(jsd_client_global):.4f}")
        
        # Pairwise JSD between clients
        jsd_pairwise = []
        for i in range(len(client_probs)):
            for j in range(i + 1, len(client_probs)):
                jsd = jensenshannon(client_probs[i], client_probs[j])
                jsd_pairwise.append(jsd)
        
        if jsd_pairwise:
            print(f"\nPairwise Client JSD (mean): {np.mean(jsd_pairwise):.4f}")
            print(f"Pairwise Client JSD (max):  {np.max(jsd_pairwise):.4f}")
            print(f"Pairwise Client JSD (min):  {np.min(jsd_pairwise):.4f}")
        
    else:
        print("\nWarning: Could not find suitable label column for heterogeneity analysis")
        print("Available columns:", [col for col in df.columns if 'Risk' in col])
else:
    print("\nWarning: 'Client' column not found. Skipping heterogeneity analysis.")

In [ ]:
# Visualize Client Heterogeneity
# Re-determine label column for visualization
label_column_viz = None
if 'Client' in df.columns:
    if 'Risk_Category_Composite' in df.columns:
        label_column_viz = 'Risk_Category_Composite'
    elif any(col.startswith('Risk_Category_') for col in df.columns):
        label_column_viz = [col for col in df.columns if col.startswith('Risk_Category_')][0]
    elif any(col.startswith('Risk_') and not col.endswith('_Prob') for col in df.columns):
        label_column_viz = [col for col in df.columns if col.startswith('Risk_') and not col.endswith('_Prob')][0]

if 'Client' in df.columns and label_column_viz:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # 1. Client sizes
    client_sizes = get_client_sizes(df, "Client")
    sizes_df = pd.DataFrame(list(client_sizes.items()), columns=['Client', 'Size'])
    axes[0].bar(sizes_df['Client'], sizes_df['Size'], alpha=0.7, color='steelblue')
    axes[0].set_xlabel('Client ID')
    axes[0].set_ylabel('Number of Samples')
    axes[0].set_title('Client Sizes (Quantity Skew)')
    axes[0].grid(True, alpha=0.3)
    
    # 2. Label distribution per client
    stats = get_label_distribution_stats(df, label_column_viz, "Client")
    all_labels = sorted(set(label for client_stats in stats.values() for label in client_stats.keys()))
    
    client_ids = sorted(stats.keys())
    label_counts_matrix = []
    for client_id in client_ids:
        client_stats = stats[client_id]
        counts = [client_stats.get(label, 0) for label in all_labels]
        label_counts_matrix.append(counts)
    
    label_counts_df = pd.DataFrame(label_counts_matrix, index=client_ids, columns=[f'Label {l}' for l in all_labels])
    label_counts_df.plot(kind='bar', stacked=True, ax=axes[1], alpha=0.8)
    axes[1].set_xlabel('Client ID')
    axes[1].set_ylabel('Count')
    axes[1].set_title(f'Label Distribution per Client ({label_column_viz})')
    axes[1].legend(title='Labels', bbox_to_anchor=(1.05, 1), loc='upper left')
    axes[1].grid(True, alpha=0.3)
    
    # 3. JSD visualization
    if 'jsd_client_global' in locals():
        axes[2].bar(range(len(jsd_client_global)), jsd_client_global, alpha=0.7, color='coral')
        axes[2].set_xlabel('Client Index')
        axes[2].set_ylabel('JSD')
        axes[2].set_title('Jensen-Shannon Divergence\n(Client vs Global Distribution)')
        axes[2].axhline(y=np.mean(jsd_client_global), color='red', linestyle='--', 
                        label=f'Mean: {np.mean(jsd_client_global):.4f}')
        axes[2].legend()
        axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("Skipping heterogeneity visualization (requires Client column and valid label column)")

In [ ]:
# Feature Distributions
print("=" * 60)
print("FEATURE DISTRIBUTIONS")
print("=" * 60)

# Demographics
demographic_cols = ['Age', 'Sex'] if 'Age' in df.columns and 'Sex' in df.columns else []
if demographic_cols:
    print("\nDemographics:")
    for col in demographic_cols:
        print(f"\n{col}:")
        print(df[col].describe())

# Categorical features
categorical_cols = [col for col in df.columns if col not in risk_cols + risk_probs + ['Client', 'Patient'] 
                   and df[col].dtype in ['int64', 'object'] and df[col].nunique() < 20]
if categorical_cols:
    print(f"\nCategorical Features ({len(categorical_cols)}):")
    for col in categorical_cols[:10]:  # Show first 10
        print(f"\n{col}:")
        print(df[col].value_counts().head())

In [ ]:
# Visualize Key Features
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

# Age distribution
if 'Age' in df.columns:
    axes[0].hist(df['Age'], bins=30, alpha=0.7, color='skyblue', edgecolor='black')
    axes[0].set_xlabel('Age')
    axes[0].set_ylabel('Frequency')
    axes[0].set_title('Age Distribution')
    axes[0].grid(True, alpha=0.3)

# Sex distribution
if 'Sex' in df.columns:
    sex_counts = df['Sex'].value_counts().sort_index()
    axes[1].bar(sex_counts.index, sex_counts.values, alpha=0.7, color=['pink', 'lightblue'])
    axes[1].set_xlabel('Sex')
    axes[1].set_ylabel('Count')
    axes[1].set_title('Sex Distribution')
    axes[1].set_xticks([0, 1])
    axes[1].set_xticklabels(['Female', 'Male'])
    axes[1].grid(True, alpha=0.3)

# Impaction Depth
if 'Impaction_Depth' in df.columns:
    depth_counts = df['Impaction_Depth'].value_counts().sort_index()
    axes[2].bar(depth_counts.index, depth_counts.values, alpha=0.7, color='coral')
    axes[2].set_xlabel('Impaction Depth')
    axes[2].set_ylabel('Count')
    axes[2].set_title('Impaction Depth Distribution')
    axes[2].grid(True, alpha=0.3)

# Risk correlation heatmap
if all(col in df.columns for col in risk_cols):
    risk_corr = df[risk_cols].corr()
    im = axes[3].imshow(risk_corr, cmap='coolwarm', aspect='auto', vmin=-1, vmax=1)
    axes[3].set_xticks(range(len(risk_cols)))
    axes[3].set_yticks(range(len(risk_cols)))
    axes[3].set_xticklabels([name.replace('Risk_', '') for name in risk_cols], rotation=45, ha='right')
    axes[3].set_yticklabels([name.replace('Risk_', '') for name in risk_cols])
    axes[3].set_title('Risk Correlation Matrix')
    plt.colorbar(im, ax=axes[3])

plt.tight_layout()
plt.show()

## Summary

This EDA notebook provides:

1. **Dataset Overview**: Basic statistics and data quality checks
2. **Risk Analysis**: Distribution of target variables and their probabilities
3. **Client Heterogeneity**: 
   - Label distribution metrics (entropy, diversity, imbalance)
   - Quantity skew metrics (CV, Gini coefficient, imbalance ratio)
   - Jensen-Shannon Divergence (JSD) for measuring distribution differences
4. **Feature Analysis**: Key feature distributions and correlations

### Key Metrics Explained

- **JSD (Jensen-Shannon Divergence)**: Measures how different client label distributions are from global distribution (0 = identical, 1 = maximally different)
- **CV (Coefficient of Variation)**: Measures quantity skew across clients (std/mean)
- **Entropy**: Measures label diversity (higher = more diverse)
- **Imbalance Ratio**: Ratio of max to min label counts (higher = more imbalanced)